## Topic: The Complete RAW Application With LangChain + Chroma DB (Correction Version)

In [ ]:
""" 
                    COMPLETE RAG APPLICATION
                    ========================

      PHASE 1 — INDEXING
      ------------------

            PDF
            │
            ▼
            PyPDFLoader
            │
            ▼
            Documents
            │
            ▼
            Text Splitter
            │
            ▼
            Chunks
            │
            ▼
            Embedding Model
            │
            ▼
            Vector Embeddings
            │
            ▼
            ChromaDB
            │
            ▼
            Persistent Vector Store
"""

In [ ]:
""" 
              PHASE 2 — RETRIEVAL + GENERATION
              ---------------------------------

User Query
    │
    ▼
Query Embedding
    │
    ▼
ChromaDB
    │
    ▼
Similarity Search
    │
    ▼
Top-K Relevant Chunks
    │
    ▼
Context + Query
    │
    ▼
Prompt
    │
    ▼
Groq LLM
    │
    ▼
Final Answer

"""

In [1]:
# --------------------------------------------------
# 1. Import Require Library
# --------------------------------------------------
import os

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(
c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Step 2 — Load environment variables
# Load variables from .env
load_dotenv()

# Check that required API key exists
if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError(
        "GROQ_API_KEY is not configured in the .env file."
    )

### Step 3 — Load PDF

In [3]:
pdf_path = "./data/ML-System-python-book.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(
        f"PDF file not found: {pdf_path}"
    )

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print(f"Loaded {len(documents)} PDF pages.")

Loaded 326 PDF pages.


### Step 4 — Split documents

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks.")

Created 656 chunks.


### Step 5 — Create embedding mode

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1053.89it/s]


### Step 6 — Create and populate ChromaDB

In [6]:
persist_directory = "./RAG_PDF_Analysis"

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="RAG_PDF",
    persist_directory=persist_directory
)

print("ChromaDB created and documents indexed successfully.")

ChromaDB created and documents indexed successfully.


### Step 7 — Create retriever

In [7]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

### Step 8 — Create the Groq model

In [8]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2
)

### Step 9 — Create RAG prompt

In [9]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful book analysis AI assistant.

Answer the user's question using ONLY the provided context.

Rules:
1. Do not use outside knowledge.
2. If the answer cannot be found in the context, say:
   "I don't know based on the provided document."
3. Keep the answer clear and concise.

Context:
{context}

Question:
{question}

Answer:
""")

### Step 10 — Format retrieved documents

In [10]:
def format_documents(documents):
    return "\n\n".join(
        document.page_content
        for document in documents
    )

### Step 11 — Build the modern RAG chain

In [11]:
rag_chain = (
    {
        "context": retriever | format_documents, # return the vector store 
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
"""  
                    User Question
                         │
             ┌───────────┴───────────┐
             ↓                       ↓
         Retriever            Passthrough
             ↓                       ↓
       Relevant Docs            Question
             ↓                       │
      format_documents              │
             ↓                       │
             └───────────┬───────────┘
                         ↓
                       Prompt
                         ↓
                    Groq LLM
                         ↓
                  StrOutputParser
                         ↓
                      Answer

"""

### Step 12 — Ask a question

In [12]:
query = "tell me About the Authors"

# Retrieve relevant documents
retrieved_docs = retriever.invoke(query)

# Build context
context = format_documents(retrieved_docs)

# Generate answer
answer = (
    prompt
    | llm
    | StrOutputParser()
).invoke({
    "context": context,
    "question": query
})

print("ANSWER")
print("=" * 60)
print(answer)

print("\nSOURCES")
print("=" * 60)

for doc in retrieved_docs:
    source = doc.metadata.get("source", "Unknown source")
    page = doc.metadata.get("page", "Unknown")

    if isinstance(page, int):
        page += 1

    print(f"Source: {source}")
    print(f"Page: {page}")
    print("-" * 60)

ANSWER
**About the Authors**

- **Luis** – Lead developer on the popular Python computer‑vision package *mahotas* and contributor to several machine‑learning codes. He splits his time between Luxembourg and Heidelberg.  
- The book’s authors are not limited to previously published writers; Packt welcomes technical experts who may have little writing experience, offering editorial support to help them develop a writing career.  
- The authors provide support and answer questions through the website www.TwoToReal.com and can be contacted via the email addresses listed in the book (e.g., author@packtpub.com, questions@packtpub.com).

SOURCES
Source: ./data/ML-System-python-book.pdf
Page: 324
------------------------------------------------------------
Source: ./data/ML-System-python-book.pdf
Page: 5
------------------------------------------------------------
Source: ./data/ML-System-python-book.pdf
Page: 20
------------------------------------------------------------


In [ ]:
""" 
                         RAG APPLICATION
                              │
             ┌────────────────┴────────────────┐
             │                                 │
             ▼                                 ▼
       INDEXING PIPELINE                 QUERY PIPELINE
       -----------------                 --------------
             │                                 │
        PDF Document                      User Query
             ↓                                 ↓
       PyPDFLoader                      Query Embedding
             ↓                                 ↓
         Documents                         Retriever
             ↓                                 ↓
     Recursive Splitter                    ChromaDB
             ↓                                 ↓
           Chunks                     Top-K Documents
             ↓                                 ↓
   HuggingFace Embeddings                      │
             ↓                                 │
          Vectors                              │
             ↓                                 │
         ChromaDB ─────────────────────────────┘
             │
             ▼
       Retrieved Context
             │
             ▼
          Prompt
             │
             ▼
      Groq GPT-OSS 20B
             │
             ▼
       Final Answer
             │
             ▼
      Source / Page Info

"""

### Personal Experiment

In [ ]:
# from dotenv import load_dotenv, find_dotenv

# _ = load_dotenv(find_dotenv())

In [ ]:
# import os
# from dotenv import load_dotenv, find_dotenv

# load_dotenv(find_dotenv())

# groq_api_key = os.getenv("GROQ_API_KEY")

# if groq_api_key:
#     print("GROQ_API_KEY found")
#     print("Key starts with:", groq_api_key[:7])
# else:
#     print("GROQ_API_KEY NOT FOUND")

GROQ_API_KEY found
Key starts with: gsk_8hR


In [ ]:
# import os

# from langchain_groq import ChatGroq

# llm = ChatGroq(
#     model="openai/gpt-oss-20b",
#     temperature=0.2,
#     api_key=os.getenv("GROQ_API_KEY")
# )

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


In [ ]:
# response = llm.invoke("Say hello in one sentence.")

# print(response.content)

Hello, how can I help you today?


In [ ]:
# import os
# from dotenv import load_dotenv

# load_dotenv()

# key = os.getenv("GROQ_API_KEY")

# print("Key found:", key is not None)
# print("Key length:", len(key) if key else 0)
# print("Starts with gsk_:", key.startswith("gsk_") if key else False)
# print("Ends with whitespace:", key != key.rstrip() if key else False)

python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 20


Key found: True
Key length: 57
Starts with gsk_: False
Ends with whitespace: False
